# 4. SKT 유입인구 분석 (뛰어야산다2 집중)

**데이터**: SKT 성연령별 유입인구 (2026.01~02)
**방송**: 뛰어야산다2 (2026-01-12 방영)
**장점**: 일별 + 읍면동별 + 성/연령별 유입 분석 가능

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import POP_DIR, read_csv_auto

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 4.1 데이터 탐색

SKT 유입인구 데이터가 매우 크므로 (9GB+) 적합한 파일 선택 필요.

**후보 파일 (작은 것부터)**:
- `AS_SKT_SX_AGE_UNQ_NOPE` (1MB) - 성연령별 순방문자
- `AS_SKT_SGG_UNQ_VST_NOPE` (678KB) - 시군구별 순방문자  
- `AS_SKT_AGE_UNQ_OUTFLOW_NOPE` (976KB) - 연령별 유출
- `AS_SKT_DOW_FLOW_NOPE` (12.6MB) - 요일별 유동인구
- `AS_SKT_SX_AGE_DONG` (49MB) - 성연령별 동 단위 (추천!)
- `AS_SKT_SX_AGE_INFLOW_NOPE` (9.2GB) - 매우 상세하지만 너무 큼

In [2]:
# 추천: AS_SKT_SGG_UNQ_VST_NOPE (시군구별 순방문자) - 가장 경량
# 또는 AS_SKT_SX_AGE_DONG (성연령별 동 단위) - 동 단위 분석 가능

# 순방문자 로드 (경량)
vst_files = sorted(POP_DIR.glob('*/AS_SKT_SGG_UNQ_VST_NOPE_*.csv'))
print(f"순방문자 파일: {len(vst_files)}개")

dfs = []
for f in vst_files:
    df = read_csv_auto(f)
    dfs.append(df)
    print(f"  {f.name}: {len(df):,}행, columns={list(df.columns)[:5]}...")

if dfs:
    vst = pd.concat(dfs, ignore_index=True)
    print(f"\n총: {vst.shape}")
    vst.head()

순방문자 파일: 8개
  AS_SKT_SGG_UNQ_VST_NOPE_202601.csv: 7,099행, columns=['CRTR_YMD', 'VST_SGG_CD', 'INFLOW_SGG_CD', 'ML_VST_PPLTN_00_19', 'ML_VST_PPLTN_20_29']...
  AS_SKT_SGG_UNQ_VST_NOPE_DAWN_202601.csv: 7,099행, columns=['CRTR_YMD', 'VST_SGG_CD', 'INFLOW_SGG_CD', 'ML_VST_PPLTN_00_19', 'ML_VST_PPLTN_20_29']...
  AS_SKT_SGG_UNQ_VST_NOPE_DAY_202601.csv: 7,099행, columns=['CRTR_YMD', 'VST_SGG_CD', 'INFLOW_SGG_CD', 'ML_VST_PPLTN_00_19', 'ML_VST_PPLTN_20_29']...
  AS_SKT_SGG_UNQ_VST_NOPE_NGHT_202601.csv: 7,099행, columns=['CRTR_YMD', 'VST_SGG_CD', 'INFLOW_SGG_CD', 'ML_VST_PPLTN_00_19', 'ML_VST_PPLTN_20_29']...
  AS_SKT_SGG_UNQ_VST_NOPE_202602.csv: 6,496행, columns=['CRTR_YMD', 'VST_SGG_CD', 'INFLOW_SGG_CD', 'ML_VST_PPLTN_00_19', 'ML_VST_PPLTN_20_29']...
  AS_SKT_SGG_UNQ_VST_NOPE_DAWN_202602.csv: 6,496행, columns=['CRTR_YMD', 'VST_SGG_CD', 'INFLOW_SGG_CD', 'ML_VST_PPLTN_00_19', 'ML_VST_PPLTN_20_29']...
  AS_SKT_SGG_UNQ_VST_NOPE_DAY_202602.csv: 6,496행, columns=['CRTR_YMD', 'VST_SGG_CD', 'INFLOW_SGG_CD

In [3]:
# 성연령별 동 단위 (더 상세)
dong_files = sorted(POP_DIR.glob('*/AS_SKT_SX_AGE_DONG_*.csv'))
print(f"성연령별 동 파일: {len(dong_files)}개")

dfs2 = []
for f in dong_files:
    df = read_csv_auto(f)
    dfs2.append(df)
    print(f"  {f.name}: {len(df):,}행")

if dfs2:
    skt_dong = pd.concat(dfs2, ignore_index=True)
    print(f"\n총: {skt_dong.shape}")
    print(f"컬럼: {list(skt_dong.columns)}")
    skt_dong.head()

성연령별 동 파일: 2개
  AS_SKT_SX_AGE_DONG_202601.csv: 1,252,509행
  AS_SKT_SX_AGE_DONG_202602.csv: 1,226,685행

총: (2479194, 6)
컬럼: ['CRTR_YMD', 'AGE', 'SX', 'DPTRE_DONG', 'ARVL_DONG', 'PPLTN_CNT']


## 4.2 뛰어야산다2 전후 유입 변화

방영일: 2026-01-12
- pre: 2026-01-01 ~ 2026-01-11
- post: 2026-01-13 ~ 2026-01-26 (2주)

In [4]:
# 날짜 컬럼 확인 후 변환
if len(dfs2) > 0:
    date_cols = [c for c in skt_dong.columns if 'YMD' in c.upper() or 'DATE' in c.upper() or 'DT' in c.upper()]
    print(f"날짜 관련 컬럼: {date_cols}")
    
    dong_cols = [c for c in skt_dong.columns if 'DONG' in c.upper() or 'STDG' in c.upper()]
    print(f"지역 관련 컬럼: {dong_cols}")
    
    pop_cols = [c for c in skt_dong.columns if 'NOPE' in c.upper() or 'POP' in c.upper() or 'CNT' in c.upper()]
    print(f"인구 관련 컬럼: {pop_cols[:10]}")

날짜 관련 컬럼: ['CRTR_YMD']
지역 관련 컬럼: ['DPTRE_DONG', 'ARVL_DONG']
인구 관련 컬럼: ['PPLTN_CNT']


In [5]:
# ★ 실행 후 컬럼명 확인하고 아래 코드 조정
# date_col = '...'  # 날짜 컬럼
# dong_col = '...'  # 읍면동 컬럼 
# pop_col = '...'   # 유입인구 컬럼

# skt_dong['date'] = pd.to_datetime(skt_dong[date_col], format='%Y%m%d')

# 촬영지 읍면동 (뛰어야산다2)
# treated = ['온천동', '염치읍']
# treated_mask = skt_dong[dong_col].apply(lambda x: any(t in str(x) for t in treated))

# daily_treated = skt_dong[treated_mask].groupby('date')[pop_col].sum()
# daily_control = skt_dong[~treated_mask].groupby('date')[pop_col].sum()

print("컬럼 확인 후 위 코드 수정하여 실행")

컬럼 확인 후 위 코드 수정하여 실행


In [6]:
# 일별 유입인구 시각화 (컬럼 확인 후)
# fig, ax = plt.subplots(figsize=(14, 5))
# daily_treated.plot(ax=ax, label='촬영 읍면동 (온천동+염치읍)', color='#e74c3c')
# daily_control.plot(ax=ax, label='비촬영 읍면동', color='#3498db', alpha=0.7)
# ax.axvline(pd.Timestamp('2026-01-12'), color='red', linestyle='--', linewidth=2, label='방영일')
# ax.legend()
# ax.set_title('뛰어야산다2 전후 SKT 유입인구')
# plt.tight_layout()
# plt.show()

print("위 시각화 코드: 컬럼 확인 후 주석 해제")

위 시각화 코드: 컬럼 확인 후 주석 해제


## 4.3 성별/연령별 유입 패턴 변화

In [7]:
# 뛰어야산다2 출연진: 션, 이영표, 양세형, 배성재 → 3040 남성 타겟
# 방송 후 3040 남성 유입 증가 여부 확인

# sex_col = '...'  # 성별 컬럼
# age_col = '...'  # 연령대 컬럼

# 방송 전후 성연령 분포 비교
# air_date = pd.Timestamp('2026-01-12')
# pre = skt_dong[(skt_dong['date'] < air_date) & treated_mask]
# post = skt_dong[(skt_dong['date'] >= air_date) & treated_mask]

# pre_demo = pre.groupby([sex_col, age_col])[pop_col].sum()
# post_demo = post.groupby([sex_col, age_col])[pop_col].sum()

print("성연령 분석: 컬럼 확인 후 실행")

성연령 분석: 컬럼 확인 후 실행


## 4.4 요일별 유입 패턴

방송 효과가 주말 방문 증가로 나타나는지 확인

In [8]:
# DOW 데이터 활용
dow_files = sorted(POP_DIR.glob('*/AS_SKT_DOW_FLOW_NOPE_*.csv'))
if dow_files:
    print(f"요일별 유동인구 파일: {len(dow_files)}개")
    # 첫 파일 구조 확인
    sample = read_csv_auto(dow_files[0], nrows=5)
    print(f"컬럼: {list(sample.columns)}")
    display(sample.head())

요일별 유동인구 파일: 2개
컬럼: ['CRTR_YM', 'SM_LC_CD', 'COORD_X', 'COORD_Y', 'FLOW_PPLTN_CNT_MON', 'FLOW_PPLTN_CNT_TUES', 'FLOW_PPLTN_CNT_WEDNES', 'FLOW_PPLTN_CNT_THURS', 'FLOW_PPLTN_CNT_FRI', 'FLOW_PPLTN_CNT_SATUR', 'FLOW_PPLTN_CNT_SUN']


,CRTR_YM,SM_LC_CD,COORD_X,COORD_Y,FLOW_PPLTN_CNT_MON,FLOW_PPLTN_CNT_TUES,FLOW_PPLTN_CNT_WEDNES,FLOW_PPLTN_CNT_THURS,FLOW_PPLTN_CNT_FRI,FLOW_PPLTN_CNT_SATUR,FLOW_PPLTN_CNT_SUN
0,202601,34040110100010000001,126.965510,36.824563,0.04,0.05,0.04,0.03,0.04,0,0
1,202601,34040110100010000001,126.965495,36.826817,0.09,0.08,0.08,0.08,0.08,0,0
2,202601,34040110100010000001,126.966156,36.812397,0.05,0.05,0.05,0.04,0.04,0,0
3,202601,34040110100010000001,126.966093,36.821411,0.03,0.02,0.02,0.02,0.02,0,0
4,202601,34040110100010000001,126.966046,36.828172,2.62,2.57,2.49,2.41,2.33,0,0
